In [1]:
import os
import collections
import numpy as np
from numpy.lib.format import open_memmap
from pathlib import Path
from tqdm import tqdm
import openwakeword
import openwakeword.data
import openwakeword.utils
import openwakeword.metrics
from openwakeword.utils import download_models
import scipy
import datasets
import matplotlib.pyplot as plt
import torch
from torch import nn
import IPython.display as ipd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
import openwakeword.data as owdata

INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
INFO:datasets:PyTorch version 2.5.1 available.


In [2]:
os.listdir()
os.getcwd()
F = openwakeword.utils.AudioFeatures(melspec_model_path="C:\Mydata\DL-WakeWord/models/melspectrogram.onnx", embedding_model_path="C:\Mydata\DL-WakeWord/models/embedding_model.onnx")

In [3]:
negative_clips, negative_durations = openwakeword.data.filter_audio_paths(
    [
    "C:/Mydata/DL-WakeWord/data/fma_sample",
    "C:/Mydata/DL-WakeWord/data/fsd50k_sample",
    "C:/Mydata/DL-WakeWord/data/cv11_test_clips"

    ],
    min_length_secs = 1.0, # minimum clip length in seconds
    max_length_secs = 60*30, # maximum clip length in seconds
    duration_method = "header" # use the file header to calculate duration
)

print(f"{len(negative_clips)} negative clips after filtering, representing ~{sum(negative_durations)//3600} hours")

200it [00:00, 14830.82it/s]
100%|██████████| 200/200 [00:00<00:00, 200.50it/s]
1000it [00:00, 169878.66it/s]
100%|██████████| 1000/1000 [00:02<00:00, 416.71it/s]
1it [00:00, 55.80it/s]
100%|██████████| 1/1 [00:00<?, ?it/s]

1097 negative clips after filtering, representing ~4.0 hours


In [4]:
lights_path = r"C:\Mydata\DL-WakeWord\data\turn_on_the_office_lights"

In [5]:
# Get positive example paths, filtering out clips that are too long or too short

positive_clips, durations = openwakeword.data.filter_audio_paths(
    [
        lights_path
    ],
    min_length_secs = 1.0, # minimum clip length in seconds
    max_length_secs = 2.0, # maximum clip length in seconds
    duration_method = "header" # use the file header to calculate duration
)

print(f"{len(positive_clips)} positive clips after filtering")

3388it [00:00, 20226.72it/s]
100%|██████████| 3388/3388 [00:04<00:00, 733.79it/s]

3203 positive clips after filtering


In [21]:
# Split dataset into 75% training and 25% testing

train_clips, test_clips, train_durations, test_durations = train_test_split(
    positive_clips, durations, test_size=0.25, random_state=10
)

train_neg_clips, test_neg_clips, train_negative_durations, test_negative_durations = train_test_split(
    negative_clips, negative_durations, test_size=0.25, random_state=10
)

In [6]:
audio_dataset_1 = datasets.Dataset.from_dict({"audio": train_neg_clips})
audio_dataset_1 = audio_dataset_1.cast_column("audio", datasets.Audio(sampling_rate=16000))
audio_dataset_2 = datasets.Dataset.from_dict({"audio": test_neg_clips})
audio_dataset_2 = audio_dataset_2.cast_column("audio", datasets.Audio(sampling_rate=16000))

In [ ]:
# Define output file paths
train_output_file = "C:/Mydata/DL-WakeWord/data/features/negative_train_features.npy"
test_output_file = "C:/Mydata/DL-WakeWord/data/features/negative_test_features.npy"

In [ ]:
# Parameters
batch_size = 64  # Number of files to load and process at a time
clip_size = 3  # Desired window size (in seconds) for the trained openWakeWord model
sample_rate = 16000  # Sample rate in Hz
clip_samples = sample_rate * clip_size  # Total samples per clip

# Compute the maximum number of rows for train and test datasets
N_total_train = int(sum(train_negative_durations) // clip_size)
N_total_test = int(sum(test_negative_durations) // clip_size)

n_feature_cols = F.get_embedding_shape(clip_size)

output_array_shape_train = (N_total_train, n_feature_cols[0], n_feature_cols[1])
output_array_shape_test = (N_total_test, n_feature_cols[0], n_feature_cols[1])

# Function to process datasets
def process_audio_dataset(audio_dataset, output_file, output_shape, dataset_name):
    # Check if the output file exists
    if os.path.exists(output_file):
        # Load the existing memory-mapped file to resume processing
        fp = open_memmap(output_file, mode='r+', dtype=np.float32, shape=output_shape)
        print(f"Existing memory-mapped file for {dataset_name} loaded.")
    else:
        # Create a new memory-mapped file if it does not exist
        fp = open_memmap(output_file, mode='w+', dtype=np.float32, shape=output_shape)
        print(f"New memory-mapped file for {dataset_name} created.")

    # Determine the starting point to resume processing
    row_counter = np.count_nonzero(fp[:, 0, 0])
    print(f"Resuming processing from row {row_counter} for {dataset_name} dataset.")

    for i in tqdm(np.arange(0, len(audio_dataset), batch_size), desc=f"Processing {dataset_name} dataset"):
        # Load data in batches and convert to the required format
        wav_data = [(clip["array"] * 32767).astype(np.int16) for clip in audio_dataset[i:i + batch_size]["audio"]]
        
        # Stack clips into a consistent rectangular shape
        wav_data = owdata.stack_clips(wav_data, clip_size=clip_samples).astype(np.int16)
        
        # Compute audio features (increase ncpu for faster processing)
        features = F.embed_clips(x=wav_data, batch_size=1024, ncpu=8)

        # Save computed features to memory-mapped array file
        if row_counter + features.shape[0] > output_shape[0]:
            fp[row_counter:output_shape[0], :, :] = features[0:output_shape[0] - row_counter, :, :]
            fp.flush()
            break
        else:
            fp[row_counter:row_counter + features.shape[0], :, :] = features
            row_counter += features.shape[0]
            fp.flush()

    # Close the memory-mapped file to release the lock
    del fp  

    # Try trimming the file after ensuring it is closed
    try:
        owdata.trim_mmap(output_file)
        print(f"Successfully trimmed {output_file}.")
    except PermissionError:
        print(f"Warning: Could not delete {output_file}. Ensure it is not in use and try again.")

    print(f"Processing completed and file saved for {dataset_name}.")

# Process train negative dataset
process_audio_dataset(audio_dataset_1, train_output_file, output_array_shape_train, "Train Negative")

# Process test negative dataset
process_audio_dataset(audio_dataset_2, test_output_file, output_array_shape_test, "Test Negative")

print("All processing completed.")

In [ ]:
# Define output file paths for train and test embeddings
train_output_file = "C:/Mydata/DL-WakeWord/data/features/pos_train_lights_embeddings.npy"
test_output_file = "C:/Mydata/DL-WakeWord/data/features/pos_test_lights_embeddings.npy"

In [26]:
audio_dataset_3 = datasets.Dataset.from_dict({"audio": test_clips})
audio_dataset_3 = audio_dataset_3.cast_column("audio", datasets.Audio(sampling_rate=16000))

In [ ]:
# Determine feature shape for memory-mapped file
n_feature_cols = F.get_embedding_shape(total_length_seconds)
output_array_shape_train = (len(train_clips), n_feature_cols[0], n_feature_cols[1])

# Create memory-mapped file for training embeddings
fp_train = open_memmap(train_output_file, mode='w+', dtype=np.float32, shape=output_array_shape_train)

row_counter = 0
for batch in tqdm(mixing_generator, total=len(train_clips)//batch_size):
    batch, lbls, background = batch[0], batch[1], batch[2]
    
    # Compute audio features
    melspecs = F._get_melspectrogram_batch(batch, batch_size=batch_size, ncpu=1)
    embeddings = F._get_embeddings_batch(melspecs[:, :, :, None], batch_size=batch_size, ncpu=1)
    
    # Save computed features
    fp_train[row_counter:row_counter+embeddings.shape[0], :, :] = embeddings
    row_counter += embeddings.shape[0]
    fp_train.flush()
    
    if row_counter >= len(train_clips):
        break

# Trim empty rows from the mmapped array
owdata.trim_mmap(train_output_file)

print("Training embeddings saved successfully!")

# Ensure all features are written to disk
fp_train.flush()
del fp_train  # Close memory-mapped array to free file access


In [ ]:
# Process test negative dataset
n_feature_cols = F.get_embedding_shape(total_length_seconds)
output_array_shape_test = (len(test_clips), n_feature_cols[0], n_feature_cols[1])
process_audio_dataset(audio_dataset_3, test_output_file, output_array_shape_test, "Test Positive")